> **Author:** Kristin Predeck  
> **Date:** April 2026  
> **Purpose:** Feature engineering for the neural network model track including CLR transform, binary presence indicators, and minimal engineered features.  
> **References:**  
> - Aitchison, J. (1986). *The Statistical Analysis of Compositional Data.* Chapman & Hall.  
> - Gal, Y. & Ghahramani, Z. (2016). Dropout as a Bayesian Approximation: Representing Model Uncertainty in Deep Learning. *ICML*.  
> **Reproducibility:** All transforms are per-row deterministic (no cross-observation statistics). Random seed 42 used where applicable. StandardScaler deferred to modeling phase.

# Feature Engineering for Neural Networks

## Overview

This notebook documents the feature engineering process for the neural network model track.

Neural networks learn feature interactions internally, so the goal here is not to hand-engineer
every ratio and interaction (as we did for XGBoost and logistic regression), but rather to
**transform the raw compositional data into a form that removes known statistical artifacts**
and provides the network with clean, unconstrained inputs.

### Key Design Decisions

1. **Centered Log-Ratio (CLR) Transform** - Our elemental composition data sums to 100% per particle
   (the "closed composition" problem identified in the sandbox notebook). This creates spurious
   correlations and constrains the feature space. The CLR transform (Aitchison, 1986) breaks this
   constraint by mapping each element to `log(x_i / geometric_mean(x))`, producing unconstrained
   real-valued features suitable for neural network input.

2. **Binary Presence Indicators** - A parallel 27-dimensional binary vector indicating whether each
   element is present (>0) or absent. This captures the sparsity structure directly and lets the
   network learn to combine presence patterns with concentration patterns.

3. **Minimal Engineered Features** - Only the highest-signal features from the XGBoost exploration
   are carried forward: the top mass-fraction ratio `(Pb+Sb)/(mass-Ba-O)` (PR-AUC 0.9988) and
   `gsr_count` (number of core GSR elements present). We keep this set small because the NN
   should learn interactions on its own.

4. **StandardScaler** - Applied after CLR to normalize scale across features, which helps with
   gradient-based optimization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## 1. Load Preprocessed Data

In [2]:
df = pd.read_parquet("../../../data/processed/preprocessed.parquet")
print(f"Shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
df.head()

Shape: (2294985, 34)
Label distribution:
label
Non_GSR    1216039
GSR        1078946
Name: count, dtype: int64


,stub_id,particle_id,relevance_class,merged_relevance_class,final_class,label,o,s,cu,ba,...,mn,as,cr,br,mo,sr,ni,w,hg,target
0,22,1454,PbSb,PbSb,PbSb,GSR,23.001350,9.189914,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,1
1,22,1274,PbSbBa,PbSbBa,PbBaSb,GSR,44.647034,10.348382,0.000000,18.079432,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,1
2,22,275,PbSbBa,PbSbBa,PbBaSb,GSR,43.118126,8.859735,0.000000,8.547903,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,1
3,22,714,PbSbBa,PbSbBa,PbBaSb,GSR,38.265804,12.629639,7.930242,11.576385,...,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,1
4,22,2887,PbSb,PbSb,PbSb,GSR,41.612095,7.725395,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,5.48293,0.0,0.0,1


In [3]:
meta_cols = [
    "stub_id",
    "particle_id",
    "relevance_class",
    "merged_relevance_class",
    "final_class",
    "label",
    "target",
]
element_cols = [c for c in df.columns if c not in meta_cols]
print(f"Element columns ({len(element_cols)}): {element_cols}")

Element columns (27): ['o', 's', 'cu', 'ba', 'al', 'si', 'ca', 'pb', 'sb', 'fe', 'zn', 'cl', 'k', 'na', 'mg', 'ti', 'sn', 'p', 'mn', 'as', 'cr', 'br', 'mo', 'sr', 'ni', 'w', 'hg']


## 2. Centered Log-Ratio (CLR) Transform

### The Problem: Closed Composition

Our elemental features sum to ~100% per particle. This is a well-known issue in compositional data analysis (Aitchison, 1986), where components are constrained to sum to a constant, and standard statistical methods produce misleading results due to:

- Correlations are spurious (if one element goes up, others must go down)
- The data lives on a simplex, not in unconstrained Euclidean space
- Neural networks assume unconstrained real-valued inputs

The sandbox notebook confirmed this: a multivariate regression of all other elements onto oxygen yielded R² = 0.9494, but the same test for lead gave R² = 0.9337.Tthe high R² values are artifacts of the sum constraint, not real predictive relationships.

### The Solution: CLR Transform

The Centered Log-Ratio transform maps each observation from the simplex to unconstrained real-valued space:

$$\text{CLR}(x_i) = \ln\left(\frac{x_i}{g(x)}\right)$$

where $g(x)$ is the geometric mean of all components. 

This:
- Breaks the sum constraint
- Preserves relative relationships between elements
- Produces features centered around zero
- Is invertible (we can recover original compositions if needed)

**Handling zeros:** Since `log(0)` is undefined, we add a small pseudocount before transforming. We use a multiplicative replacement strategy where zeros are replaced with a small value (half the minimum non-zero value per element) and the remaining values are adjusted to
maintain the sum constraint.

In [4]:
def multiplicative_replacement(X, delta=None):
    """
    For each zero entry, replace with delta (default: half the minimum non-zero
    value in that column). Non-zero entries are scaled down proportionally so
    rows still sum to the same total.
    """
    X = X.copy().astype(float)

    if delta is None:
        nonzero_vals = X[X > 0]
        delta = nonzero_vals.min() / 2 if len(nonzero_vals) > 0 else 1e-10

    row_sums = X.sum(axis=1, keepdims=True)

    for i in range(X.shape[0]):
        zero_mask = X[i] == 0
        n_zeros = zero_mask.sum()
        if n_zeros > 0 and row_sums[i, 0] > 0:
            X[i, zero_mask] = delta
            correction = 1 - (n_zeros * delta / row_sums[i, 0])
            X[i, ~zero_mask] *= correction

    return X


def clr_transform(X):
    """Centered Log-Ratio transform."""
    log_X = np.log(X)
    geometric_mean = log_X.mean(axis=1, keepdims=True)
    return log_X - geometric_mean

In [5]:
X_raw = df[element_cols].values

print(f"Zero rate before replacement: {(X_raw == 0).mean():.4f}")
print(f"Row sums (first 5): {X_raw.sum(axis=1)[:5]}")

Zero rate before replacement: 0.7098
Row sums (first 5): [ 99.99999833 100.00000358 100.0000003  100.00000066  99.99999762]


In [6]:
X_replaced = multiplicative_replacement(X_raw)
print(f"\nZero rate after replacement: {(X_replaced == 0).mean():.4f}")
print(f"Row sums after replacement (first 5): {X_replaced.sum(axis=1)[:5]}")


Zero rate after replacement: 0.0000
Row sums after replacement (first 5): [ 99.99999833 100.00000358 100.0000003  100.00000066  99.99999762]


In [7]:
X_clr = clr_transform(X_replaced)
print(f"\nCLR shape: {X_clr.shape}")
print(f"CLR row sums (should be ~0): {X_clr.sum(axis=1)[:5]}")


CLR shape: (2294985, 27)
CLR row sums (should be ~0): [ 3.01980663e-14  1.59872116e-14 -1.42108547e-14 -5.32907052e-15
  0.00000000e+00]


In [8]:
clr_cols = [f"clr_{c}" for c in element_cols]
df_clr = pd.DataFrame(X_clr, columns=clr_cols, index=df.index)

### CLR Validation

Compare the correlation structure before and after CLR to confirm the transform is working as intended.

In [ ]:
# using just a random sample for resource constrained compujte
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(df), size=50_000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

raw_corr = df[element_cols].iloc[sample_idx].corr()
mask = np.triu(np.ones_like(raw_corr, dtype=bool))
sns.heatmap(
    raw_corr,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[0],
    cbar_kws={"shrink": 0.7},
)
axes[0].set_title("Raw Element Correlations (closed composition)", fontsize=12)

clr_corr = df_clr.iloc[sample_idx].corr()
mask2 = np.triu(np.ones_like(clr_corr, dtype=bool))
sns.heatmap(
    clr_corr,
    mask=mask2,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=axes[1],
    cbar_kws={"shrink": 0.7},
)
axes[1].set_title("CLR-Transformed Correlations (open composition)", fontsize=12)

plt.tight_layout()
plt.savefig(
    "outputs/feature_exploration_nn__raw_element_correlations_closed_composition.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 3. Binary Presence Indicators

The sparsity analysis in EDA showed that most elements are zero for most particles. A binary presence vector captures this structure directly and gives the network a separate channel for "which elements are present" vs. "how much of each element."

This is especially useful for GSR classification because the *combination* of elements present (e.g., Pb + Ba + Sb together) is as diagnostic as their concentrations.

In [10]:
presence_cols = [f"has_{c}" for c in element_cols]
df_presence = pd.DataFrame(
    (df[element_cols].values > 0).astype(int), columns=presence_cols, index=df.index
)

print(f"Presence features shape: {df_presence.shape}")
print(f"\nMean presence rates (top 10):")
print(df_presence.mean().sort_values(ascending=False).head(10))

Presence features shape: (2294985, 27)

Mean presence rates (top 10):
has_o     0.997824
has_s     0.803313
has_ba    0.699774
has_cu    0.676917
has_al    0.594304
has_si    0.550235
has_ca    0.447888
has_pb    0.424135
has_zn    0.409889
has_sb    0.373467
dtype: float64


## 4. Minimal Engineered Features

We carry forward only the highest-signal engineered features from the XGBoost exploration. The NN should learn most interactions on its own, so we keep this set deliberately small:

- `pb_sb_over_non_ba_o_mass`: (Pb+Sb) / (total_mass - Ba - O), PR-AUC 0.9988 in XGBoost exploration
- `log_pb_plus_sb`: log1p(Pb + Sb), PR-AUC 0.9973
- `gsr_count`: count of {Pb, Ba, Sb} present, simple but effective domain feature

In [11]:
df_eng = pd.DataFrame(index=df.index)
total_mass = df[element_cols].sum(axis=1)
denom = total_mass - df["ba"] - df["o"]
safe_denom = np.where(denom == 0, -1, denom)
df_eng["pb_sb_over_non_ba_o_mass"] = (df["pb"] + df["sb"]) / safe_denom
df_eng["log_pb_plus_sb"] = np.log1p(df["pb"] + df["sb"])
df_eng["gsr_count"] = (
    (df["pb"] > 0).astype(int) + (df["ba"] > 0).astype(int) + (df["sb"] > 0).astype(int)
)

print(f"Engineered features shape: {df_eng.shape}")
print(f"\nFeature summary:")
print(df_eng.describe().round(4))

Engineered features shape: (2294985, 3)

Feature summary:
       pb_sb_over_non_ba_o_mass  log_pb_plus_sb     gsr_count
count              2.294985e+06    2.294985e+06  2.294985e+06
mean               2.399000e-01    1.359200e+00  1.497400e+00
std                3.124000e-01    1.538300e+00  1.070000e+00
min                0.000000e+00    0.000000e+00  0.000000e+00
25%                0.000000e+00    0.000000e+00  1.000000e+00
50%                0.000000e+00    0.000000e+00  1.000000e+00
75%                5.021000e-01    2.941600e+00  2.000000e+00
max                1.000000e+00    4.615100e+00  3.000000e+00


## 5. Assemble Final Feature Set

| Group | Features | Count | Purpose |
|-------|----------|-------|---------|
| CLR-transformed elements | `clr_o`, `clr_s`, ... `clr_hg` | 27 | Unconstrained compositional representation |
| Binary presence flags | `has_o`, `has_s`, ... `has_hg` | 27 | Sparsity structure / element co-occurrence |
| Engineered features | `pb_sb_over_non_ba_o_mass`, `log_pb_plus_sb`, `gsr_count` | 3 | High-signal domain features |
| **Total** | | **57** | |

In [12]:
df_nn = pd.concat(
    [
        df[["stub_id", "particle_id", "label", "target", "final_class"]],
        df_clr,
        df_presence,
        df_eng,
    ],
    axis=1,
)

feature_cols = clr_cols + presence_cols + list(df_eng.columns)
print(f"Total features:{len(feature_cols)}")
print(f"Final dataset shape: {df_nn.shape}")
print(f"\nFeature groups:")
print(f"CLR elements:{len(clr_cols)}")
print(f"Presence flags:{len(presence_cols)}")
print(f"Engineered: {len(df_eng.columns)}")
print(f"\nNaN count:{df_nn[feature_cols].isna().sum().sum()}")
print(f"Inf count:{np.isinf(df_nn[feature_cols].values).sum()}")

Total features:57
Final dataset shape: (2294985, 62)

Feature groups:
CLR elements:27
Presence flags:27
Engineered: 3

NaN count:0
Inf count:0


## 6. Distribution Inspection

In [ ]:
core_elements = ["pb", "ba", "sb"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, elem in enumerate(core_elements):
    ax = axes[0, i]
    for label, color in [("GSR", "crimson"), ("Non_GSR", "steelblue")]:
        vals = df[df["label"] == label][elem]
        ax.hist(
            vals[vals > 0], bins=80, density=True, alpha=0.5, color=color, label=label
        )
    ax.set_title(f"Raw {elem.upper()} (non-zero only)")
    ax.legend(fontsize=8)

    ax = axes[1, i]
    for label, color in [("GSR", "crimson"), ("Non_GSR", "steelblue")]:
        mask = df["label"] == label
        ax.hist(
            df_clr.loc[mask, f"clr_{elem}"],
            bins=80,
            density=True,
            alpha=0.5,
            color=color,
            label=label,
        )
    ax.set_title(f"CLR {elem.upper()}")
    ax.legend(fontsize=8)

plt.suptitle("Raw vs CLR Distributions for Core GSR Elements", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(
    "outputs/feature_exploration_nn__raw_elem_upper_non_zero_only.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 7. Save Engineered Dataset

In [ ]:
# df_nn.to_parquet("../../../data/processed/engineered_features_nn.parquet", index=False)
print(f"Saved: {df_nn.shape[0]:,} rows x {df_nn.shape[1]} columns")
print(f"Feature columns: {len(feature_cols)}")